In [14]:
panels = {
    "CGN":   ["C3","CD46","CFH","CFHR5","CFI","COL4A3","COL4A4","COL4A5","COL4A6","FN1"],
    "CAKUT": ["BMP4","CHD1L","DSTYK","EYA1","GATA3","HNF1B","MUC1","PAX2","RET","ROBO2",
              "SALL1","SIX1","SIX2","SIX5","SOX17","SRGAP1","TBX18","TNXB","UMOD","UPK3A",
              "WNT4","ACE","AGT","AGTR1","CHRM3","FGF20","FRAS1","FREM1","FREM2","GRIP1",
              "HPSE2","ITGA8","LRIG2","REN","TRAP1","KAL1"],
    "SRNS":  ["ADCK4","ARHGDIA","CD2AP","COQ2","COQ6","CRB2","DGKE","EMP2",
              "ITGA3","ITGB4","KANK1","KANK2","KANK4","LAMB2","MYO1E","NPHS1","NPHS2",
              "NUP93","NUP107","NUP205","PDSS2","PLCE1","PTPRO","SCARB2","SMARCAL1",
              "WDR73","XPO5","ACTN4","ANLN","ARHGAP24","INF2","LMX1B","MYH9","TRPC6","WT1"],
    "USD":   ["ADCY10","AGXT","APRT","ATP6V0A4","ATP6V1B1","CA2","CASR","CLCN5","CLCNKB",
              "CLDN16","CLDN19","CYP24A1","FAM20A","GRHPR","HNF4A","HOGA1","HPRT1","KCNJ1",
              "OCRL","SLC12A1","SLC22A12","SLC2A9","SLC34A1","SLC34A3","SLC3A1","SLC4A1",
              "SLC7A9","VDR","XDH"],
    "NPHP":  ["NPHP1","INVS","NPHP3","NPHP4","IQCB1","CEP290","GLIS2","RPGRIP1L","NEK8",
              "SDCCAG8","TMEM67","TTC21B","WDR19","ZNF423","CEP164","ANKS6"],
}

seen, all_genes = set(), []
for panel, genes in panels.items():
    for gene in genes:
        if gene not in seen:
            all_genes.append((gene, panel))
            seen.add(gene)

print(f"Total: {len(all_genes)} genes")
for panel, genes in panels.items():
    print(f"  {panel}: {len(genes)}")

Total: 126 genes
  CGN: 10
  CAKUT: 36
  SRNS: 35
  USD: 29
  NPHP: 16


In [15]:
import requests, time, pandas as pd

search_url = ("https://rest.uniprot.org/uniprotkb/search"
              "?query=gene_exact:{gene}+AND+organism_id:9606+AND+reviewed:true"
              "&fields=accession,gene_names,length,protein_name&format=json")

rows = []
for i, (gene, panel) in enumerate(all_genes, 1):
    row = {"gene": gene, "panel": panel, "accession": None,
           "length": None, "entry_name": None, "flag": ""}
    try:
        r    = requests.get(search_url.format(gene=gene), timeout=15)
        hits = r.json().get("results", [])
        row["flag"] = "" if len(hits) == 1 else ("no_hit" if not hits else f"multi({len(hits)})")
        if hits:
            row["accession"]  = hits[0]["primaryAccession"]
            row["length"]     = hits[0].get("sequence", {}).get("length")
            row["entry_name"] = hits[0].get("uniProtkbId", "")
        print(f"[{i:03d}] {gene:<12} {row['accession'] or '—':>10}  {str(row['length'] or ''):>6} aa  {row['flag']}")
    except Exception as e:
        row["flag"] = f"error: {e}"
        print(f"[{i:03d}] {gene:<12} error: {e}")
    rows.append(row)
    time.sleep(0.15)

acc_df = pd.DataFrame(rows)
acc_df.to_csv("/content/uniprot_accessions.csv", index=False)
print(f"\nResolved: {acc_df['accession'].notna().sum()}/{len(acc_df)}")
print(acc_df[acc_df["flag"] != ""][["gene","accession","flag"]])

[001] C3               P01024    1663 aa  
[002] CD46             P15529     392 aa  
[003] CFH              P08603    1231 aa  
[004] CFHR5            Q9BXR6     569 aa  
[005] CFI              P05156     583 aa  
[006] COL4A3           Q01955    1670 aa  
[007] COL4A4           P53420    1690 aa  
[008] COL4A5           P29400    1685 aa  
[009] COL4A6           Q14031    1691 aa  
[010] FN1              P02751    2477 aa  
[011] BMP4             P12644     408 aa  
[012] CHD1L            Q86WJ1     897 aa  
[013] DSTYK            Q6XUX3     929 aa  
[014] EYA1             Q99502     592 aa  
[015] GATA3            P23771     443 aa  
[016] HNF1B            P35680     557 aa  
[017] MUC1             P15941    1255 aa  
[018] PAX2             Q02962     417 aa  
[019] RET              P07949    1114 aa  
[020] ROBO2            Q9HCK4    1378 aa  
[021] SALL1            Q9NSC2    1324 aa  
[022] SIX1             Q15475     284 aa  
[023] SIX2             Q9NPC8     291 aa  
[024] SIX5 

In [16]:
import requests, time, re, pandas as pd
from pathlib import Path

af_api  = "https://alphafold.ebi.ac.uk/api/prediction/{}"
out_dir = Path("/content/structures")
out_dir.mkdir(exist_ok=True)

acc_df = pd.read_csv("/content/uniprot_accessions.csv")
log = []

for i, row in enumerate(acc_df.itertuples(), 1):
    entry = {"gene": row.gene, "panel": row.panel, "accession": row.accession,
             "status": None, "n_fragments": 0, "max_uniprot_end": None,
             "uniprot_length": row.length, "note": ""}

    if pd.isna(row.accession):
        entry["status"] = "no_uniprot"
        print(f"[{i:03d}] {row.gene:<12} — skipped (no accession)")
        log.append(entry)
        continue

    try:
        r = requests.get(af_api.format(row.accession), timeout=20)
        if r.status_code == 404:
            entry["status"] = "no_af_model"
            print(f"[{i:03d}] {row.gene:<12} {row.accession}  — not in AlphaFold DB")
            log.append(entry)
            time.sleep(0.1)
            continue
        r.raise_for_status()
        fragments = r.json()

        # Only accept canonical fragments: AF-{accession}-F{n} — no isoform infix.
        canonical_pattern = re.compile(rf"^AF-{re.escape(row.accession)}-F(\d+)$")

        saved = []
        max_uniprot_end = 0
        for frag in fragments:
            entry_id = frag.get("entryId", "")
            m = canonical_pattern.match(entry_id)
            if not m:
                entry["note"] += f"skipped non-canonical fragment '{entry_id}'; "
                continue
            f_num = m.group(1)
            fname = f"{row.gene}.pdb" if len(fragments) == 1 else f"{row.gene}_F{f_num}.pdb"
            pr    = requests.get(frag["pdbUrl"], timeout=60)
            pr.raise_for_status()
            if pr.content[:5] in (b"HEADE", b"REMAR", b"ATOM ", b"MODEL"):
                # Prepend TITLE record with all-caps gene name
                title_line = f"TITLE     {row.gene.upper():<70}\n".encode()
                pdb_text = title_line + pr.content
                (out_dir / fname).write_bytes(pdb_text)
                saved.append(fname)
                max_uniprot_end = max(max_uniprot_end, frag.get("uniprotEnd") or 0)
            time.sleep(0.05)

        entry["n_fragments"]    = len(saved)
        entry["max_uniprot_end"] = max_uniprot_end
        entry["status"]        = "ok" if saved else "failed"

        coverage_note = ""
        if saved and row.length and max_uniprot_end < row.length:
            coverage_note = f"  ⚠ TRUNCATED: covers {max_uniprot_end}/{row.length}"
            entry["note"] += f"truncated: {max_uniprot_end}/{row.length}; "

        frag_str = f"  ({len(fragments)} API frags → {len(saved)} canonical)" if len(fragments) != len(saved) else ""
        print(f"[{i:03d}] {row.gene:<12} {row.accession}  ✓{frag_str}{coverage_note}")

    except Exception as e:
        entry["status"] = "error"
        entry["note"]   = str(e)
        print(f"[{i:03d}] {row.gene:<12} error: {e}")

    log.append(entry)
    time.sleep(0.2)

log_df = pd.DataFrame(log)
log_df.to_csv("/content/download_log.csv", index=False)
print(f"\n{log_df['status'].value_counts().to_string()}")
print(f"Files saved: {len(list(out_dir.glob('*.pdb')))}")

# ── zip all PDB structures ────────────────────────────────────────────────────
import zipfile
zip_path = Path("/content/structures.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for pdb in sorted(out_dir.glob("*.pdb")):
        zf.write(pdb, arcname=pdb.name)
print(f"Zipped {len(list(out_dir.glob('*.pdb')))} PDB files → {zip_path.name} "
      f"({zip_path.stat().st_size / 1024:.1f} KB)")

truncated = log_df[log_df["note"].str.contains("truncated", na=False)]
if len(truncated):
    print(f"\nStill truncated ({len(truncated)}):")
    print(truncated[["gene","accession","max_uniprot_end","uniprot_length"]].to_string(index=False))

[001] C3           P01024  ✓
[002] CD46         P15529  ✓  (16 API frags → 1 canonical)
[003] CFH          P08603  ✓  (2 API frags → 1 canonical)
[004] CFHR5        Q9BXR6  ✓
[005] CFI          P05156  ✓
[006] COL4A3       Q01955  ✓  (5 API frags → 1 canonical)
[007] COL4A4       P53420  ✓
[008] COL4A5       P29400  ✓  (2 API frags → 1 canonical)
[009] COL4A6       Q14031  ✓  (2 API frags → 1 canonical)
[010] FN1          P02751  ✓  (17 API frags → 1 canonical)
[011] BMP4         P12644  ✓
[012] CHD1L        Q86WJ1  ✓  (5 API frags → 1 canonical)
[013] DSTYK        Q6XUX3  ✓  (4 API frags → 1 canonical)
[014] EYA1         Q99502  ✓  (3 API frags → 1 canonical)
[015] GATA3        P23771  ✓  (2 API frags → 1 canonical)
[016] HNF1B        P35680  ✓  (4 API frags → 1 canonical)
[017] MUC1         P15941  ✓  (17 API frags → 1 canonical)
[018] PAX2         Q02962  ✓  (4 API frags → 1 canonical)
[019] RET          P07949  ✓  (2 API frags → 1 canonical)
[020] ROBO2        Q9HCK4  ✓  (3 API fra

In [17]:
# ── strip injected TITLE lines (mkdssp v4 requires HEADER first) ─────────────
fixed = 0
for pdb in sorted(out_dir.glob("*.pdb")):
    data = pdb.read_bytes()
    if data.startswith(b"TITLE"):
        pdb.write_bytes(data[data.index(b"\n") + 1:])
        fixed += 1
print(f"Stripped TITLE from {fixed} PDB files")

Stripped TITLE from 123 PDB files


In [18]:
import requests, time, pandas as pd

acc_df = pd.read_csv("/content/uniprot_accessions.csv")
acc_df = acc_df[acc_df["accession"].notna()]

keep_types = {
    "Domain", "Region", "Active site", "Binding site", "Site",
    "Signal", "Propeptide", "Transit peptide", "Chain",
    "Transmembrane", "Coiled coil", "Compositional bias",
    "Modified residue", "Disulfide bond", "Motif",
}

rows = []
for i, row in enumerate(acc_df.itertuples(), 1):
    gene, acc = row.gene, row.accession
    try:
        r = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.json", timeout=20)
        r.raise_for_status()
        data     = r.json()
        features = data.get("features", [])
        length   = data.get("sequence", {}).get("length")
        n = 0
        for f in features:
            ftype = f.get("type", "")
            if ftype not in keep_types:
                continue
            loc   = f.get("location", {})
            start = loc.get("start", {}).get("value")
            end   = loc.get("end",   {}).get("value")
            rows.append({
                "gene": gene, "panel": row.panel, "accession": acc, "length": length,
                "feature_type": ftype, "description": f.get("description", ""),
                "start": start, "end": end,
            })
            n += 1
        print(f"[{i:03d}] {gene:<12} {acc}  {length:>5} aa  {n} features")
    except Exception as e:
        print(f"[{i:03d}] {gene:<12} {acc}  ERROR: {e}")
    time.sleep(0.15)

feat_df = pd.DataFrame(rows)
feat_df.to_csv("/content/uniprot_features.csv", index=False)
print(f"\nTotal feature rows: {len(feat_df)}")
print(feat_df["feature_type"].value_counts().to_string())

[001] C3           P01024   1663 aa  47 features
[002] CD46         P15529    392 aa  24 features
[003] CFH          P08603   1231 aa  62 features
[004] CFHR5        Q9BXR6    569 aa  29 features
[005] CFI          P05156    583 aa  45 features
[006] COL4A3       Q01955   1670 aa  45 features
[007] COL4A4       P53420   1690 aa  46 features
[008] COL4A5       P29400   1685 aa  43 features
[009] COL4A6       Q14031   1691 aa  41 features
[010] FN1          P02751   2477 aa  89 features
[011] BMP4         P12644    408 aa  9 features
[012] CHD1L        Q86WJ1    897 aa  20 features
[013] DSTYK        Q6XUX3    929 aa  9 features
[014] EYA1         Q99502    592 aa  14 features
[015] GATA3        P23771    443 aa  14 features
[016] HNF1B        P35680    557 aa  11 features
[017] MUC1         P15941   1255 aa  29 features
[018] PAX2         Q02962    417 aa  6 features
[019] RET          P07949   1114 aa  65 features
[020] ROBO2        Q9HCK4   1378 aa  27 features
[021] SALL1        Q9NS

In [19]:
import subprocess
subprocess.run(["apt-get", "install", "-q", "-y", "dssp"], check=True)
print("Done.")

Done.


In [20]:
import subprocess, pandas as pd
from pathlib import Path

out_dir = Path("/content/structures")
rsa_dir = Path("/content/rsa")
rsa_dir.mkdir(exist_ok=True)

max_asa = {
    "ALA":129.0,"ARG":274.0,"ASN":195.0,"ASP":193.0,"CYS":167.0,
    "GLN":225.0,"GLU":223.0,"GLY":104.0,"HIS":224.0,"ILE":197.0,
    "LEU":201.0,"LYS":236.0,"MET":224.0,"PHE":240.0,"PRO":159.0,
    "SER":155.0,"THR":172.0,"TRP":285.0,"TYR":263.0,"VAL":174.0,
}

ss_map = {
    "H":"helix","G":"helix","I":"helix",
    "E":"sheet","B":"sheet",
    "T":"loop","S":"loop"," ":"loop","-":"loop",
}

aa3_map = {
    "A":"ALA","R":"ARG","N":"ASN","D":"ASP","C":"CYS","Q":"GLN",
    "E":"GLU","G":"GLY","H":"HIS","I":"ILE","L":"LEU","K":"LYS",
    "M":"MET","F":"PHE","P":"PRO","S":"SER","T":"THR","W":"TRP",
    "Y":"TYR","V":"VAL",
}

def parse_dssp(text):
    rows = []
    in_data = False
    for line in text.splitlines():
        if "#  RESIDUE AA STRUCTURE" in line:
            in_data = True
            continue
        if not in_data:
            continue
        if len(line) < 38:
            continue
        if line[13] == "!":
            continue
        try:
            resnum = int(line[5:10].strip())
            aa     = line[13].strip()
            ss_raw = line[16]
            asa    = float(line[35:38].strip())
        except (ValueError, IndexError):
            continue
        rows.append({"resnum": resnum, "aa": aa, "ss_raw": ss_raw,
                     "ss": ss_map.get(ss_raw, "loop"), "asa": asa})
    return rows

def extract_plddt(pdb_path):
    plddt = {}
    with open(pdb_path) as f:
        for line in f:
            if line[:4] == "ATOM" and line[12:16].strip() == "CA":
                try:
                    resnum       = int(line[22:26].strip())
                    plddt[resnum] = float(line[60:66].strip())
                except ValueError:
                    continue
    return plddt

log = []
pdb_files = sorted(out_dir.glob("*.pdb"))
print(f"Running DSSP on {len(pdb_files)} files...\n")

for i, pdb_path in enumerate(pdb_files, 1):
    stem  = pdb_path.stem
    gene  = stem.split("_F")[0]
    f_num = stem.split("_F")[1] if "_F" in stem else "1"

    try:
        result = subprocess.run(
            ["mkdssp", "--output-format", "dssp", str(pdb_path)],
            capture_output=True, text=True, timeout=120
        )
        if result.returncode != 0:
            raise RuntimeError(result.stderr.strip()[:200])

        dssp_rows = parse_dssp(result.stdout)
        if not dssp_rows:
            raise RuntimeError("no residues parsed")

        plddt_map = extract_plddt(pdb_path)

        rows = []
        for r in dssp_rows:
            rn      = r["resnum"]
            resname = aa3_map.get(r["aa"])
            max_a   = max_asa.get(resname) if resname else None
            rsa     = round(r["asa"] / max_a, 4) if (r["asa"] is not None and max_a) else None
            plddt   = plddt_map.get(rn)
            rows.append({
                "gene":     gene,
                "fragment": f_num,
                "resnum":   rn,
                "aa":       r["aa"],
                "ss_raw":   r["ss_raw"],
                "ss":       r["ss"],
                "asa":      r["asa"],
                "rsa":      rsa,
                "plddt":    round(plddt, 2) if plddt is not None else None,
            })

        df = pd.DataFrame(rows)
        df.to_csv(rsa_dir / f"{stem}.csv", index=False)
        print(f"[{i:03d}] {stem:<22} {len(rows):>5} residues")
        log.append({"file": stem, "gene": gene, "fragment": f_num,
                    "residues": len(rows), "status": "ok"})

    except Exception as e:
        print(f"[{i:03d}] {stem:<22} ERROR: {e}")
        log.append({"file": stem, "gene": gene, "fragment": f_num,
                    "residues": 0, "status": f"error: {e}"})

pd.DataFrame(log).to_csv("/content/dssp_log.csv", index=False)
print(f"\nDone.")
print(pd.DataFrame(log)["status"].value_counts().to_string())

Running DSSP on 123 files...

[001] ACE_F1                  1306 residues
[002] ACTN4_F1                 911 residues
[003] ADCK4_F1                 544 residues
[004] ADCY10_F1               1610 residues
[005] AGT                      476 residues
[006] AGTR1                    359 residues
[007] AGXT                     392 residues
[008] ANKS6_F1                 871 residues
[009] ANLN_F1                 1124 residues
[010] APRT_F1                  180 residues
[011] ARHGAP24_F1              748 residues
[012] ARHGDIA_F1               204 residues
[013] ATP6V0A4                 840 residues
[014] ATP6V1B1                 513 residues
[015] BMP4                     408 residues
[016] C3                      1663 residues
[017] CA2                      260 residues
[018] CASR_F1                 1078 residues
[019] CD2AP                    639 residues
[020] CD46_F1                  392 residues
[021] CEP164_F1               1460 residues
[022] CEP290                  2479 residues
[0

In [21]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

rsa_dir  = Path("/content/rsa")
full_dir = Path("/content/rsa_full")
full_dir.mkdir(exist_ok=True)

gene_files = defaultdict(list)
for f in sorted(rsa_dir.glob("*.csv")):
    gene = f.stem.split("_F")[0]
    gene_files[gene].append(f)

for gene, files in sorted(gene_files.items()):
    frames = [pd.read_csv(f) for f in sorted(files)]
    if len(frames) == 1:
        df = frames[0]
    else:
        df = (pd.concat(frames)
                .drop_duplicates(subset="resnum", keep="first")
                .sort_values("resnum")
                .reset_index(drop=True))
    df.to_csv(full_dir / f"{gene}.csv", index=False)
    print(f"{gene:<12} {len(files)} frag(s)  →  {len(df)} residues")

print(f"\nFull-sequence CSVs saved to {full_dir}/")

ACE          1 frag(s)  →  1306 residues
ACTN4        1 frag(s)  →  911 residues
ADCK4        1 frag(s)  →  544 residues
ADCY10       1 frag(s)  →  1610 residues
AGT          1 frag(s)  →  476 residues
AGTR1        1 frag(s)  →  359 residues
AGXT         1 frag(s)  →  392 residues
ANKS6        1 frag(s)  →  871 residues
ANLN         1 frag(s)  →  1124 residues
APRT         1 frag(s)  →  180 residues
ARHGAP24     1 frag(s)  →  748 residues
ARHGDIA      1 frag(s)  →  204 residues
ATP6V0A4     1 frag(s)  →  840 residues
ATP6V1B1     1 frag(s)  →  513 residues
BMP4         1 frag(s)  →  408 residues
C3           1 frag(s)  →  1663 residues
CA2          1 frag(s)  →  260 residues
CASR         1 frag(s)  →  1078 residues
CD2AP        1 frag(s)  →  639 residues
CD46         1 frag(s)  →  392 residues
CEP164       1 frag(s)  →  1460 residues
CEP290       1 frag(s)  →  2479 residues
CFH          1 frag(s)  →  1231 residues
CFHR5        1 frag(s)  →  569 residues
CFI          1 frag(s)  →  583 r

In [22]:
import pandas as pd, json, zipfile
from pathlib import Path

full_dir  = Path("/content/rsa_full")
feat_df   = pd.read_csv("/content/uniprot_features.csv")
acc_df    = pd.read_csv("/content/uniprot_accessions.csv")
html_dir  = Path("/content/html/biophysical/genes")
html_dir.mkdir(parents=True, exist_ok=True)

template = """<!DOCTYPE html>
<html lang="en"><head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>{gene} — NephVar Biophysical</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;500;600;700&family=DM+Mono:wght@300;400&family=Instrument+Serif:ital@0;1&display=swap" rel="stylesheet">
<style>
:root{{--serif:'Instrument Serif',Georgia,serif;--sans:'Syne',system-ui,sans-serif;
--mono:'DM Mono',ui-monospace,monospace;--accent:#1f6fa8;--ink:#16202b;
--soft:#5d6b78;--faint:#8a96a3;--line:#e6e9ee;--bg:#fbfaf7;
--helix:#e07b18;--sheet:#1f6fa8;--loop:#c8ced4;
--buried:#c0392b;--exposed:#2ecc71}}
*{{box-sizing:border-box}}
body{{margin:0;font-family:var(--sans);color:var(--ink);background:var(--bg);-webkit-font-smoothing:antialiased}}
.wrap{{max-width:1240px;margin:0 auto;padding:40px 22px 70px}}
.crumb{{font-family:var(--mono);font-size:11px;letter-spacing:.14em;text-transform:uppercase;color:var(--faint);margin-bottom:16px}}
.crumb a{{color:var(--accent);text-decoration:none}}.crumb a:hover{{text-decoration:underline}}
h1{{font-family:var(--serif);font-weight:400;font-size:40px;margin:0 0 4px}}
h1 em{{font-style:italic;color:var(--accent)}}
.sub{{color:var(--soft);font-size:14px;margin-bottom:28px;line-height:1.6}}
.map-wrap{{background:#fff;border:1px solid var(--line);border-radius:6px;padding:24px;margin-bottom:32px;overflow-x:auto}}
.map-title{{font-family:var(--mono);font-size:10.5px;text-transform:uppercase;letter-spacing:.1em;color:var(--soft);margin-bottom:16px}}
svg.protmap{{display:block;width:100%;min-width:600px;height:180px}}
.legend{{display:flex;flex-wrap:wrap;gap:16px;margin-top:14px;font-family:var(--mono);font-size:10.5px;color:var(--soft)}}
.legend span{{display:flex;align-items:center;gap:6px}}
.swatch{{width:12px;height:12px;border-radius:2px;display:inline-block;flex-shrink:0}}
.controls{{display:flex;flex-wrap:wrap;gap:10px;align-items:center;margin-bottom:14px}}
input[type=text],select{{font-family:var(--sans);font-size:13px;padding:9px 12px;border:1px solid var(--line);border-radius:4px;background:#fff;color:var(--ink)}}
input[type=text]{{flex:1;min-width:200px}}
.count{{font-family:var(--mono);font-size:11px;color:var(--faint);margin-left:auto}}
table{{width:100%;border-collapse:collapse;font-size:13px;background:#fff;border:1px solid var(--line);border-radius:4px;overflow:hidden}}
th{{font-family:var(--mono);text-align:left;padding:11px 12px;background:#f6f5f1;font-weight:400;font-size:10.5px;letter-spacing:.1em;text-transform:uppercase;color:var(--soft);cursor:pointer;white-space:nowrap}}
th:hover{{color:var(--ink)}}
td{{padding:9px 12px;border-top:1px solid var(--line)}}
tbody tr:hover td{{background:#f7f9fb}}
.mono{{font-family:var(--mono);font-size:11.5px}}
.pill{{font-family:var(--mono);display:inline-block;padding:2px 8px;border-radius:99px;font-size:10px;color:#fff}}
.pill.helix{{background:var(--helix)}}.pill.sheet{{background:var(--sheet)}}.pill.loop{{background:#8a96a3}}
.pill.buried{{background:var(--buried)}}.pill.exposed{{background:var(--exposed)}}
.empty{{padding:40px;text-align:center;color:var(--faint)}}
#tip{{position:fixed;display:none;background:#16202b;color:#fff;font-family:var(--mono);
font-size:11px;padding:8px 12px;border-radius:4px;pointer-events:none;z-index:20;line-height:1.6}}
</style></head>
<body><div class="wrap">
<div class="crumb"><a href="../">NephVar</a> / <a href="./">Biophysical</a> / {gene}</div>
<h1>{gene} <em>{protein_name}</em></h1>
<div class="sub">{panel} panel &middot; {length} aa &middot;
UniProt <a href="https://www.uniprot.org/uniprot/{accession}" target="_blank"
style="color:var(--accent)">{accession}</a></div>

<div class="map-wrap">
  <div class="map-title">Protein map — domains / secondary structure / RSA / pLDDT</div>
  <svg class="protmap" id="protmap" viewBox="0 0 1000 180"></svg>
  <div class="legend">
    <span><span class="swatch" style="background:var(--helix)"></span>Helix</span>
    <span><span class="swatch" style="background:var(--sheet)"></span>Sheet</span>
    <span><span class="swatch" style="background:var(--loop)"></span>Loop / coil</span>
    <span><span class="swatch" style="background:#a0c4e8"></span>pLDDT confidence</span>
    <span><span class="swatch" style="background:var(--buried)"></span>Buried (RSA &lt; 0.25)</span>
    <span><span class="swatch" style="background:var(--exposed)"></span>Exposed (RSA &ge; 0.25)</span>
  </div>
</div>

<div class="controls">
  <input type="text" id="q" placeholder="Search position or amino acid…">
  <select id="ss"><option value="">All secondary structures</option>
    <option value="helix">Helix</option>
    <option value="sheet">Sheet</option>
    <option value="loop">Loop / coil</option>
  </select>
  <select id="dom"><option value="">All domains</option>{domain_options}</select>
  <span class="count" id="count"></span>
</div>
<table><thead><tr>
  <th data-k="resnum">Pos</th>
  <th data-k="aa">AA</th>
  <th data-k="rsa">RSA</th>
  <th data-k="ss">2° structure</th>
  <th data-k="plddt">pLDDT</th>
  <th data-k="buried">Burial</th>
  <th data-k="domain">Domain</th>
</tr></thead><tbody id="rows"></tbody></table>
<div class="empty" id="empty" style="display:none">No residues match.</div>
</div>
<div id="tip"></div>

<script>
const residues = {residue_json};
const domains  = {domain_json};
const seqlen   = {length};

const svg  = document.getElementById("protmap");
const ns   = "http://www.w3.org/2000/svg";
const W    = 1000, pad = 20, inner = W - pad * 2;
const xOf  = r => pad + ((r - 1) / Math.max(seqlen - 1, 1)) * inner;

const yDom = 12, domH = 14;
const ySS  = 36, ssH  = 12;
const yRSA = 62, rsaH = 32;
const yPLD = 108, pldH = 32;

function rect(attrs) {{
  const e = document.createElementNS(ns, "rect");
  for (const [k,v] of Object.entries(attrs)) e.setAttribute(k,v);
  return e;
}}
function text(attrs, label) {{
  const e = document.createElementNS(ns, "text");
  for (const [k,v] of Object.entries(attrs)) e.setAttribute(k,v);
  e.textContent = label;
  return e;
}}

[["Domains", yDom], ["2° Str.", ySS], ["RSA", yRSA], ["pLDDT", yPLD]].forEach(([label, y]) =>
  svg.appendChild(text({{x:2, y:y-1, fill:"#8a96a3", "font-size":"7",
    "font-family":"DM Mono,monospace"}}, label))
);

const domColors = ["#7ec8e3","#f4a261","#a8dadc","#c77dff","#90be6d",
                   "#f9c74f","#f94144","#43aa8b","#577590","#e9c46a"];
const domIdx = {{}};
domains.forEach(d => {{
  if (!d.start || !d.end) return;
  if (!(d.description in domIdx)) domIdx[d.description] = domColors[Object.keys(domIdx).length % domColors.length];
  const x = xOf(d.start), w = Math.max(2, xOf(d.end) - xOf(d.start));
  const r = rect({{x, y:yDom, width:w, height:domH, fill:domIdx[d.description], opacity:"0.85", rx:"2"}});
  const t = document.createElementNS(ns, "title");
  t.textContent = `${{d.description}} (${{d.start}}–${{d.end}})`;
  r.appendChild(t);
  svg.appendChild(r);
}});

residues.forEach(r => {{
  const x  = xOf(r.resnum);
  const ss = r.ss || "loop";
  const ssColor = ss === "helix" ? "#e07b18" : ss === "sheet" ? "#1f6fa8" : "#c8ced4";
  svg.appendChild(rect({{x, y:ySS, width:"1.5", height:ssH, fill:ssColor}}));

  if (r.rsa != null) {{
    const h = r.rsa * rsaH;
    svg.appendChild(rect({{x, y:yRSA+rsaH-h, width:"1.5", height:h,
      fill: r.rsa < 0.25 ? "#c0392b" : "#2ecc71", opacity:"0.7"}}));
  }}
  if (r.plddt != null) {{
    const h = (r.plddt / 100) * pldH;
    svg.appendChild(rect({{x, y:yPLD+pldH-h, width:"1.5", height:h, fill:"#a0c4e8", opacity:"0.8"}}));
  }}
}});

const typePriority = {{"Domain": 0, "Motif": 1, "Compositional bias": 2, "Region": 3}};

function domainOf(resnum) {{
  const hits = domains.filter(d => resnum >= d.start && resnum <= d.end);
  if (!hits.length) return "";
  hits.sort((a, b) => (typePriority[a.type] ?? 9) - (typePriority[b.type] ?? 9)
                    || (a.end - a.start) - (b.end - b.start));
  return hits[0].description;
}}

const rows = residues.map(r => ({{
  ...r,
  domain: domainOf(r.resnum),
  buried: r.rsa == null ? "" : r.rsa < 0.25 ? "buried" : "exposed"
}}));

let sortKey = "resnum", sortAsc = true;
const tbody   = document.getElementById("rows");
const countEl = document.getElementById("count");
const emptyEl = document.getElementById("empty");

function render() {{
  const q   = document.getElementById("q").value.toLowerCase();
  const ss  = document.getElementById("ss").value;
  const dom = document.getElementById("dom").value;

  let data = rows.filter(r =>
    (!q   || String(r.resnum).includes(q) || (r.aa||"").toLowerCase().includes(q)) &&
    (!ss  || r.ss === ss) &&
    (!dom || r.domain === dom)
  );

  data.sort((a, b) => {{
    const av = a[sortKey] ?? "", bv = b[sortKey] ?? "";
    return sortAsc ? (av > bv ? 1 : -1) : (av < bv ? 1 : -1);
  }});

  tbody.innerHTML = data.map(r => `
    <tr>
      <td class="mono">${{r.resnum}}</td>
      <td class="mono">${{r.aa || ""}}</td>
      <td class="mono">${{r.rsa != null ? r.rsa.toFixed(3) : "—"}}</td>
      <td><span class="pill ${{r.ss}}">${{r.ss}}</span></td>
      <td class="mono">${{r.plddt != null ? r.plddt.toFixed(1) : "—"}}</td>
      <td><span class="pill ${{r.buried}}">${{r.buried || "—"}}</span></td>
      <td style="color:var(--soft);font-size:12px">${{r.domain}}</td>
    </tr>`).join("");

  countEl.textContent = `${{data.length.toLocaleString()}} residues`;
  emptyEl.style.display = data.length ? "none" : "block";
}}

document.querySelectorAll("th[data-k]").forEach(th =>
  th.addEventListener("click", () => {{
    if (sortKey === th.dataset.k) sortAsc = !sortAsc;
    else {{ sortKey = th.dataset.k; sortAsc = true; }}
    render();
  }})
);
["q","ss","dom"].forEach(id =>
  document.getElementById(id).addEventListener("input", render)
);
render();
</script></body></html>"""

# ── generate all gene pages ──────────────────────────────────────────────────
generated = []
for rsa_path in sorted(full_dir.glob("*.csv")):
    gene    = rsa_path.stem
    acc_row = acc_df[acc_df["gene"] == gene]
    if acc_row.empty:
        print(f"{gene}: no accession — skipping"); continue

    r            = acc_row.iloc[0]
    accession    = r["accession"]
    panel        = r["panel"]
    length       = int(r["length"]) if pd.notna(r.get("length")) else 0
    protein_name = r.get("entry_name", "")
    protein_name = "" if pd.isna(protein_name) else str(protein_name).replace("_HUMAN","")

    rsa_df       = pd.read_csv(rsa_path)
    residue_data = rsa_df[["resnum","aa","ss","rsa","plddt"]].to_dict(orient="records")

    gene_feats   = feat_df[feat_df["gene"] == gene]
    domain_data  = (gene_feats[["feature_type","description","start","end"]]
                    .rename(columns={"feature_type":"type"})
                    .dropna(subset=["start","end"])
                    .to_dict(orient="records"))
    domain_names = sorted(gene_feats[gene_feats["feature_type"].isin(
        ["Domain","Region","Compositional bias","Motif"])]["description"]
        .dropna().unique())
    domain_options = "\n".join(f'<option value="{d}">{d}</option>' for d in domain_names)

    html = template.format(
        gene=gene, protein_name=protein_name, panel=panel,
        accession=accession, length=length,
        domain_options=domain_options,
        residue_json=json.dumps(residue_data),
        domain_json=json.dumps(domain_data),
    )

    out_path = html_dir / f"{gene}.html"
    out_path.write_text(html, encoding="utf-8")
    generated.append(out_path)

print(f"Generated {len(generated)} gene pages → {html_dir}/\n")

# ── split into 3 zip files ───────────────────────────────────────────────────
n          = len(generated)
batch_size = -(-n // 3)
batches    = [generated[i:i+batch_size] for i in range(0, n, batch_size)]

# ── pack all gene pages into a single zip ────────────────────────────────────
zip_dir  = Path("/content/zips")
zip_dir.mkdir(exist_ok=True)
zip_path = zip_dir / "nephvar_biophysical_genes.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in generated:
        zf.write(f, arcname=f"genes/{f.name}")

print(f"Zipped {len(generated)} gene pages → {zip_path.name} "
      f"({zip_path.stat().st_size / 1024:.1f} KB)")

for i, batch in enumerate(batches, 1):
    zip_path = zip_dir / f"nephvar_biophysical_genes_batch{i}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in batch:
            zf.write(f, arcname=f"genes/{f.name}")
    print(f"Batch {i}: {len(batch)} files → {zip_path.name} "
          f"({zip_path.stat().st_size / 1024:.1f} KB)")
    print(f"   genes: {', '.join(f.stem for f in batch)}")

Generated 123 gene pages → /content/html/biophysical/genes/

Zipped 123 gene pages → nephvar_biophysical_genes.zip (1642.3 KB)
Batch 1: 41 files → nephvar_biophysical_genes_batch1.zip (540.5 KB)
   genes: ACE, ACTN4, ADCK4, ADCY10, AGT, AGTR1, AGXT, ANKS6, ANLN, APRT, ARHGAP24, ARHGDIA, ATP6V0A4, ATP6V1B1, BMP4, C3, CA2, CASR, CD2AP, CD46, CEP164, CEP290, CFH, CFHR5, CFI, CHD1L, CHRM3, CLCN5, CLCNKB, CLDN16, CLDN19, COL4A3, COL4A4, COL4A5, COL4A6, COQ2, COQ6, CRB2, CYP24A1, DGKE, DSTYK
Batch 2: 41 files → nephvar_biophysical_genes_batch2.zip (578.6 KB)
   genes: EMP2, EYA1, FAM20A, FGF20, FN1, FREM1, GATA3, GLIS2, GRHPR, GRIP1, HNF1B, HNF4A, HOGA1, HPRT1, HPSE2, INF2, INVS, IQCB1, ITGA3, ITGA8, ITGB4, KAL1, KANK1, KANK2, KANK4, KCNJ1, LAMB2, LMX1B, LRIG2, MUC1, MYH9, MYO1E, NEK8, NPHP1, NPHP3, NPHP4, NPHS1, NPHS2, NUP107, NUP205, NUP93
Batch 3: 41 files → nephvar_biophysical_genes_batch3.zip (523.2 KB)
   genes: OCRL, PAX2, PDSS2, PLCE1, PTPRO, REN, RET, ROBO2, RPGRIP1L, SALL1, SCARB2,

In [23]:
from google.colab import files
for i in range(1, 4):
    files.download(f"/content/zips/nephvar_biophysical_genes_batch{i}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>